# Consultas SQL principales — TechZone BigQuery

Notebook preparado para validar y analizar el modelo de datos de e-commerce en BigQuery.

Incluye las **6 consultas más importantes** para demostrar que el modelo permite análisis de negocio:

1. Ventas totales.
2. Evolución mensual de ventas.
3. Ventas por categoría.
4. Productos más vendidos.
5. Clientes que más han gastado.
6. Margen estimado por producto.

In [19]:
from google.cloud import bigquery
import os
from IPython.display import display, Markdown, HTML

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../credentials/key.json"

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
BQ_DATASET_ID = os.getenv("BQ_DATASET_ID")

print("PROJECT_ID:", PROJECT_ID)
print("BQ_DATASET_ID:", BQ_DATASET_ID)

client = bigquery.Client(project=PROJECT_ID)

PROJECT_ID: techzone-494713
BQ_DATASET_ID: TechZone


## Funciones auxiliares

`mostrar_kpi()` se usa para resultados de un solo valor.

`mostrar_resultado()` se usa para consultas que devuelven tablas con varias filas o columnas.

In [13]:
def formatear_euros(valor):
    """Formatea un número como importe en euros con formato español."""
    if valor is None:
        return "0,00"
    return f"{float(valor):,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def mostrar_kpi(titulo, valor, descripcion=None, prefijo="", sufijo=""):
    html = f"""
    <div style="
        border: 1px solid #ddd;
        border-radius: 12px;
        padding: 20px;
        margin: 15px 0;
        background-color: #f8f9fa;
        max-width: 420px;
        font-family: Arial, sans-serif;
    ">
        <h3 style="margin: 0 0 10px 0; color: #333;">{titulo}</h3>
        <div style="
            font-size: 32px;
            font-weight: bold;
            color: #1f77b4;
            margin-bottom: 8px;
        ">
            {prefijo}{valor}{sufijo}
        </div>
        <p style="margin: 0; color: #666; font-size: 14px;">
            {descripcion if descripcion else ""}
        </p>
    </div>
    """
    display(HTML(html))


def mostrar_resultado(titulo, df, descripcion=None):
    display(Markdown(f"### {titulo}"))
    if descripcion:
        display(Markdown(descripcion))
    display(df)

## 1. Ventas totales

Esta consulta calcula el importe neto vendido teniendo en cuenta:

```text
(quantity * purchase_unit_price) - discount_amount
```

Es una métrica básica para saber cuánto ha vendido el e-commerce en total.

In [14]:
query_total_sales = f"""
SELECT
  ROUND(SUM((quantity * purchase_unit_price) - discount_amount), 2) AS total_sales
FROM `{PROJECT_ID}.{BQ_DATASET_ID}.order_items`
"""

df_total_sales = client.query(query_total_sales).to_dataframe()

total_sales = df_total_sales.loc[0, "total_sales"]

mostrar_kpi(
    "Ventas totales",
    formatear_euros(total_sales),
    "Importe total vendido calculado a partir de las líneas de pedido.",
    sufijo=" €"
)

c:\Users\csancho\Documents\GitHub\TheBridge\Personal\TC-sql-TechZone\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 2. Evolución mensual de ventas

Esta consulta permite ver cómo evolucionan las ventas mes a mes.

In [15]:
query_monthly_sales = f"""
SELECT
  FORMAT_TIMESTAMP('%Y-%m', o.order_date) AS order_month,
  ROUND(SUM((oi.quantity * oi.purchase_unit_price) - oi.discount_amount), 2) AS total_sales,
  COUNT(DISTINCT o.order_id) AS total_orders
FROM `{PROJECT_ID}.{BQ_DATASET_ID}.orders` o
JOIN `{PROJECT_ID}.{BQ_DATASET_ID}.order_items` oi
  ON o.order_id = oi.order_id
GROUP BY order_month
ORDER BY order_month
"""

df_monthly_sales = client.query(query_monthly_sales).to_dataframe()

mostrar_resultado(
    "Evolución mensual de ventas",
    df_monthly_sales,
    "Ventas agrupadas por mes junto con el número de pedidos realizados."
)

### Evolución mensual de ventas

Ventas agrupadas por mes junto con el número de pedidos realizados.

,order_month,total_sales,total_orders
0,2023-05,1817.13,2
1,2023-06,5422.02,1
2,2023-07,20076.66,8
3,2023-08,11246.63,5
4,2023-09,22685.00,7
5,2023-10,29984.37,13
6,2023-11,45606.63,14
7,2023-12,37274.95,13
8,2024-01,36783.40,17
9,2024-02,72631.55,28


## 3. Ventas por categoría

Esta consulta une `order_items`, `products` y `categories` para calcular qué categorías generan más ingresos.

In [16]:
query_sales_by_category = f"""
SELECT
  c.name AS category_name,
  ROUND(SUM((oi.quantity * oi.purchase_unit_price) - oi.discount_amount), 2) AS total_sales,
  SUM(oi.quantity) AS units_sold
FROM `{PROJECT_ID}.{BQ_DATASET_ID}.order_items` oi
JOIN `{PROJECT_ID}.{BQ_DATASET_ID}.products` p
  ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{BQ_DATASET_ID}.categories` c
  ON p.category_id = c.category_id
GROUP BY category_name
ORDER BY total_sales DESC
"""

df_sales_by_category = client.query(query_sales_by_category).to_dataframe()

mostrar_resultado(
    "Ventas por categoría",
    df_sales_by_category,
    "Ranking de categorías según el importe total vendido."
)

c:\Users\csancho\Documents\GitHub\TheBridge\Personal\TC-sql-TechZone\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


### Ventas por categoría

Ranking de categorías según el importe total vendido.

,category_name,total_sales,units_sold
0,Laptops,1931399.22,1662
1,Smartphones,1119020.17,1634
2,Tablets,803394.11,1563
3,Wearables,599586.39,1597
4,Gaming,348022.40,1384
5,Audio,292825.24,1649
6,Periféricos,200216.36,1514
7,Almacenamiento,174042.28,1362


## 4. Productos más vendidos

Esta consulta identifica los productos con más unidades vendidas.

In [17]:
query_top_products_units = f"""
SELECT
  p.product_id,
  p.name AS product_name,
  c.name AS category_name,
  SUM(oi.quantity) AS total_units_sold,
  ROUND(SUM((oi.quantity * oi.purchase_unit_price) - oi.discount_amount), 2) AS total_sales
FROM `{PROJECT_ID}.{BQ_DATASET_ID}.order_items` oi
JOIN `{PROJECT_ID}.{BQ_DATASET_ID}.products` p
  ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{BQ_DATASET_ID}.categories` c
  ON p.category_id = c.category_id
GROUP BY p.product_id, product_name, category_name
ORDER BY total_units_sold DESC
LIMIT 10
"""

df_top_products_units = client.query(query_top_products_units).to_dataframe()

mostrar_resultado(
    "Productos más vendidos",
    df_top_products_units,
    "Top 10 productos ordenados por unidades vendidas."
)

c:\Users\csancho\Documents\GitHub\TheBridge\Personal\TC-sql-TechZone\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


### Productos más vendidos

Top 10 productos ordenados por unidades vendidas.

,product_id,product_name,category_name,total_units_sold,total_sales
0,12,Dell XPS T21,Laptops,216,325433.08
1,6,Motorola K42,Smartphones,210,143509.17
2,19,Bose QC I16,Audio,208,15162.81
3,36,Huawei Watch F81,Wearables,208,45757.58
4,38,Razer DeathAdder C26,Periféricos,207,65722.56
5,24,JBL Tune M79,Audio,207,38639.26
6,35,Huawei Watch I63,Wearables,202,103016.03
7,20,Sony WH U69,Audio,201,14060.01
8,27,Jabra Elite S72,Audio,198,20923.02
9,21,AirPods B31,Audio,198,66976.57


## 5. Clientes que más han gastado

Esta consulta permite detectar los clientes más valiosos para el negocio.

In [18]:
query_top_customers = f"""
SELECT
  c.customer_id,
  CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
  c.email,
  c.country,
  c.city,
  COUNT(DISTINCT o.order_id) AS total_orders,
  ROUND(SUM((oi.quantity * oi.purchase_unit_price) - oi.discount_amount), 2) AS total_spent
FROM `{PROJECT_ID}.{BQ_DATASET_ID}.customers` c
JOIN `{PROJECT_ID}.{BQ_DATASET_ID}.orders` o
  ON c.customer_id = o.customer_id
JOIN `{PROJECT_ID}.{BQ_DATASET_ID}.order_items` oi
  ON o.order_id = oi.order_id
GROUP BY c.customer_id, customer_name, c.email, c.country, c.city
ORDER BY total_spent DESC
LIMIT 10
"""

df_top_customers = client.query(query_top_customers).to_dataframe()

mostrar_resultado(
    "Clientes que más han gastado",
    df_top_customers,
    "Top 10 clientes ordenados por importe total gastado."
)

c:\Users\csancho\Documents\GitHub\TheBridge\Personal\TC-sql-TechZone\venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


### Clientes que más han gastado

Top 10 clientes ordenados por importe total gastado.

,customer_id,customer_name,email,country,city,total_orders,total_spent
0,197,Jaime Vives,aagusti@example.net,Portugal,Ávila,12,41024.90
1,388,Reinaldo Arcos,xiomarabru@example.com,Países Bajos,Toledo,9,34079.97
2,426,Obdulia Quesada,dionisio98@example.org,Alemania,Cuenca,6,31931.86
3,92,Rosalva Arnal,dalfonso@example.org,España,Albacete,6,31750.86
4,50,Luís Cases,acalderon@example.net,Portugal,Lugo,7,31151.31
5,199,Lorenza Garcés,zairarequena@example.net,España,Huelva,8,31028.04
6,278,Chelo Porcel,ariadnamaestre@example.net,Países Bajos,Valencia,7,30493.04
7,28,Antonio Camps,uojeda@example.com,Países Bajos,Teruel,9,29685.02
8,433,Mariana Moreno,melaniacanet@example.com,Alemania,Lugo,10,29485.77
9,302,Basilio Delgado,agata28@example.com,Italia,Ceuta,4,29015.07
